# Facility Recommender — Quick Tour

A guided tour of the facility recommender built for the Virtue Foundation dataset (DAIS 2026). The goal of the project: **given a patient's symptoms + location, recommend nearby facilities and justify each recommendation with verified evidence.**

This notebook shows:
1. **Where the data lives** — the tables the recommender is built on
2. **How confident we are** — the confidence-band idea in one chart
3. **Two live demos** — toothache near Pune, chest pain near Delhi — with plain-English explanations of each ranking

Everything here runs against `labuser_jdw2` gold/platinum tables rebuilt by the full pipeline (medallion → features → catalog → evidence → ontology → confidence).

## 1. Where everything lives

One row per **facility** (10,077 of them), then one row per **facility × service** pair stating how well-evidenced that service is. Run the cell below to see the layer cake.

In [0]:
# One-time setup: make the project's src/ importable and load the
# recommender module (needs extraction, geo filter, ranking, explanations).
import sys
sys.path.insert(0, "/Workspace/Users/jdw2@andrew.cmu.edu/cmu_datathon26/src")

import recommend  # from cmu_datathon26/src: extract_needs, geo_candidates,
                 # rank, explain, render_explanation, recommend
print("recommend module loaded:", recommend.__file__)


In [0]:
tables = [
    ("gold.facility_district_health_indicators",
     "10,077 facilities + NFHS-5 district health context (the base dataset)"),
    ("platinum.facility_class",
     "facility type: hospital / clinic / dental_clinic / diagnostic_center / ..."),
    ("platinum.facility_scale",
     "doctor counts and bed capacity where the scrape provides them"),
    ("platinum.service_catalog",
     "the 192 canonical services (101 specialties + 91 procedures/equipment)"),
    ("platinum.facility_service_evidence",
     "who claims what, through which channel (specialty claim, procedure, equipment, capability, description)"),
    ("platinum.corroboration_rules",
     "what should support a believable claim (e.g. bypass surgery needs an operating theatre + ICU + blood bank)"),
    ("platinum.facility_service_confidence",
     "one confidence band per facility × service pair — the heart of the system"),
]
for t, desc in tables:
    n = spark.table(f"labuser_jdw2.{t}").count()
    print(f"{n:>9,}  labuser_jdw2.{t:<45s} {desc}")

## 2. How confident are we in each claim?

A facility *says* it offers things in five ways: its **specialty list**, the **procedures** it describes, the **equipment** it lists, its **capability** text, and its marketing **description**. A service claimed through only one weak channel is less believable than one corroborated across channels — and claims that violate the facility's scale (a 1-doctor clinic claiming neurosurgery) get flagged.

The bands:
* **high** — corroborated across ≥2 channels including a strong one, all required support present, scale-consistent
* **medium** — believable but not fully corroborated (e.g. strong single channel, or a specialty claim at a substantial facility)
* **unverified** — thin or weak evidence only
* **low** — a scale violation (facility class or doctor count below the service's floor); this is the *only* path to low

In [0]:
display(spark.sql("""
  SELECT confidence_band, COUNT(*) AS pairs,
         ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
  FROM labuser_jdw2.platinum.facility_service_confidence
  GROUP BY confidence_band
  ORDER BY pairs DESC
"""))

## 3. How a recommendation works (4 steps)
1. **Understand the patient** — an LLM turns free-text symptoms + context into a weighted list of needs from the fixed 192-service catalog (unknown service names are dropped, never guessed); the most safety-critical needs are marked *critical*
2. **Filter by distance** — facilities within the radius (haversine); pincode-centroid coordinates are flagged and gently penalized, never silently treated as exact
3. **Rank** — score = Σ weight × confidence-band score (high 1.0 / medium 0.5 / unverified 0.3 / low 0.05). **Critical needs are hard filters**: a facility without good evidence on a critical need is excluded, no matter how good the rest looks. Ties break by NABH accreditation → beds → facility class → distance
4. **Explain** — every recommendation cites its evidence: which channels corroborate each service, how many mentions, and honestly lists needs with *no evidence found (unknown ≠ disproven)*

The two demos below run this end-to-end. Each takes ~10–20 seconds (one LLM call for the needs extraction).

### Demo 1 — toothache near Pune

In [0]:
needs, ranked = recommend.recommend(
    spark, lat=18.5204, lng=73.8567, radius_km=25,
    symptoms="severe toothache, swollen gum",
    context="35-year-old male")
rows = ranked.collect()
exps = recommend.explain(spark, needs, rows)

print("What the model understood the patient needs:")
for n in sorted(needs, key=lambda x: -x["weight"]):
    tag = "CRITICAL" if n["critical"] else "supporting"
    print(f"  {n['service']:<24s} weight {n['weight']:.2f}  ({tag})")
print()
for f in exps[:3]:
    print(recommend.render_explanation(f))
    print()

### Demo 2 — chest pain near Delhi (a critical-needs case)

Chest pain makes `emergencyCare` a *critical* need, so any facility without solid emergency evidence is excluded outright — that's why this top-5 looks different from the toothache one.

In [0]:
needs, ranked = recommend.recommend(
    spark, lat=28.6139, lng=77.2090, radius_km=25,
    symptoms="chest pain and shortness of breath, sweating",
    context="58-year-old male, smoker, diabetic, on metformin")
rows = ranked.collect()
exps = recommend.explain(spark, needs, rows)

for n in sorted(needs, key=lambda x: -x["weight"]):
    tag = "CRITICAL" if n["critical"] else "supporting"
    print(f"  {n['service']:<24s} weight {n['weight']:.2f}  ({tag})")
print()
for f in exps[:3]:
    print(recommend.render_explanation(f))
    print()

## 4. How to read the output — and what NOT to over-trust

Every explanation line cites **verified evidence only**: the band, the channels that corroborate it, and mention counts. "No evidence found for X (unknown, not disproven)" means the scrape simply never mentioned X — it is not a statement that the facility lacks X.

Important caveats (full list in `cmu_datathon26/QA_REPORT.md`):
* Confidence is **internal-consistency of the facility's own scrape**, not clinical-outcome quality — never present it as a guarantee
* 722 facilities have empty procedure lists, 2,178 empty equipment lists (thin scrapes cap at medium/unverified rather than being penalized)
* Fully-corroborated facilities are genuinely scarce for high-acuity surgery (e.g. only 6 facilities have high-band CABG evidence) — the recommender honestly shows medium bands there instead of inflating confidence